In [7]:
import yaml
from types import SimpleNamespace
from strategies.run_allen import get_allen_result, get_allen_signals
from strategies.run_gino import get_gino_result, get_gino_signals
from strategies.utils.analysis import get_equal_weight_baseline_result
from utils.io import read_all_df
import pandas as pd
from evaluation.stats import sharpe
import finlab
from finlab import data

finlab.login('ntSS3778pZi2FfkeYxXP0p+S0iI4AggkcphAUxh/lTVrWqT2FreKQsDkTA92CM7d#vip_m')

CATEGGORY = 'Example'
result_dir = "results/Example_Result"

def load_config(path):
    with open(path, "r") as f:
        cfg_dict = yaml.safe_load(f)
    return SimpleNamespace(**cfg_dict)
backtest_config = load_config("config/backtest.yaml")

f = open("config/backtest.yaml")
cfg = yaml.safe_load(f)

result = get_allen_result(cfg, result_dir)
signals = get_allen_signals(cfg, result_dir)
test_pred, test_truth, _, _ = read_all_df(result_dir)

輸入成功!
輸入成功!


In [8]:
baseline_results = get_equal_weight_baseline_result(result_dir, pd.to_datetime('2021-01-01'), pd.to_datetime('2024-12-31'))

In [9]:
def yearly_returns(returns):
    yearly_roi = returns.resample("Y").last() / returns.resample("Y").first() - 1
    rois = {}
    for year in yearly_roi.index:
        rois[year.year] = yearly_roi[year]
    rois['final'] = returns.iloc[-1] / returns.iloc[0] - 1
    rois['sharpe'] = sharpe(returns)
    return rois

In [10]:
model_rois = yearly_returns(result['model'].returns)
baseline_rois = yearly_returns(baseline_results.returns)

print("model rois =", model_rois)
print("baseline rois =", baseline_rois)

model rois = {2021: 0.8111415542595026, 2022: -0.0902881268215634, 2023: 0.8251302244247187, 2024: 0.13551291121339082, 'final': 2.404023703779772, 'sharpe': 1.2378640828935898}
baseline rois = {2021: 0.5232284919379941, 2022: -0.16634200083612227, 2023: 0.5337833037633546, 2024: 0.1654033351804125, 'final': 1.2736038076304665, 'sharpe': 1.0840031193828383}


In [11]:
from evaluation.surge_metrics import compute_surge_metrics_from_buy_signals

metric = compute_surge_metrics_from_buy_signals(
    test_pred=test_pred,
    test_truth=test_truth,
    buy_signals=signals['buy_signals'],
    threshold=0.3
)

model_data = {
    'category': CATEGGORY,
    'baseline': 0,
    'precision': metric['actual_surge_rate_on_buy_signals'],
}
model_data.update(model_rois)

baseline_data = {
    'category': CATEGGORY,
    'baseline': 1,
    'precision': metric['overall_actual_surge_rate'],
}
baseline_data.update(baseline_rois)

result_df = pd.DataFrame([model_data, baseline_data])
result_df


,category,baseline,precision,2021,2022,2023,2024,final,sharpe
0,Example,0,0.142955,0.811142,-0.090288,0.825130,0.135513,2.404024,1.237864
1,Example,1,0.069795,0.523228,-0.166342,0.533783,0.165403,1.273604,1.084003


In [12]:
model_df = pd.DataFrame([model_data])
baseline_df = pd.DataFrame([baseline_data])

display_df = pd.DataFrame(index=model_df.index)
display_df['category'] = model_df['category']

for col in model_df.columns:
    if col == 'category':
        continue
    model_values = model_df[col]
    baseline_values = baseline_df[col]
    delta = model_values - baseline_values
    if str(col) in ['2021', '2022', '2023', '2024', 'final']:
        display_df[col] = model_values.mul(100).round(1).apply(lambda x: f"{x:.0f}%") + \
                        delta.mul(100).round(1).apply(lambda x: f"({x:+.0f}%)")

    else:
        display_df[col] = model_values.round(2).astype(str) + \
                            delta.round(2).apply(lambda x: f" ({x:+.2f})")
                            
display_df = display_df.drop(columns=['baseline'])
display_df

,category,precision,2021,2022,2023,2024,final,sharpe
0,Example,0.14 (+0.07),81%(+29%),-9%(+8%),82%(+29%),14%(-3%),240%(+113%),1.24 (+0.15)
